# Research Proposal
___
___

**Arbitrage-Free Generative Option Surfaces for Systematic Covered Call and Put Overlay Strategies**

*This proposal builds a pipeline that fits real option surfaces with SANOS, a non-parametric method guaranteeing smooth, strictly arbitrage-free prices via convex Black-Scholes kernels, then extends it into a dynamic generative model (DYSANOS) by evolving a low-dimensional latent surface state through a mean-reverting process jointly with spot. The resulting realistic, always arbitrage-free simulated paths train a Deep Hedging reinforcement-learning agent to learn optimal covered call/put rebalancing (timing, sizing, strike selection) against a chosen risk objective (e.g. CVaR), rather than static Greeks-based rules. Open challenges include multi-asset extensions, adjusting for American-style early exercise, and efficient time-series modeling of the latent state.*

**Keywords**: Option Surface Modeling, Deep Hedging, Reinforcement Learning, Time Series Analysis, Generative Modeling, Systematic Options Overlay 

___
### **Research Overview**
___

### Deep Hedging

H. Buehler, L. Gonon, J. Teichmann, and B. Wood. Deep hedging. Quantitative Finance, 19(8):1271–1291, 2019.
https://arxiv.org/abs/1802.03042 

*Deep hedging trains a reinforcement learning agent to learn how to optimize losses (or pnl) against a certain risk objective, on simulated asset prices.*



##### *Base Scenario*

Consider the following situation. Assume the market has no friction,  simulated paths follow Black-Scholes. The hedging has only the stock price as a trading instrument. 

Let $S_t$ be the underlying stock. You are short a European call option. At each rebalancing step $t$, you hold $\delta_t$ shares of the underlying. The P&L of the hedged portfolio is:

$$
\small
\text{PnL} = \text{Option Premium} - \text{Payoff}(S_T) + \sum_{t=0}^{T-1} \delta_t (S_{t+1} - S_t) - \text{Transaction Cost}
$$

The goal is to learn $\delta_t = f_\theta(\text{state}_t)$ that minimizes a risk measure (e.g. CVaR) of the terminal P&L.

In this case, the Black-Scholes hedging will be delta-hedging. Assuming trading is discrete, hold of size $\delta_t = \Phi(d_1(S_t, t))$ the asset $S_t$ at each rebalancing step $t \leq T$

In the case of the deep learning agent, we will use a neural netwrok to approximate the rebalancing rule function $\delta_t = f_\theta(\text{state}_t)$

Consider the following Neural Network:
- 2 Inputs: Asset price and Time to Maturity
- 2 Hidden Layers, 64 neurons per Layers *(arbitrary)*
- 1 output (delta)
- ReLU activation function: $\text{ReLU}(x) = \text{max}(x, 0)$
- Sigmoid output activation function: $\sigma(x) = \frac{1}{1+e^{-x}} \in [0,1]$

In this base case example, basically, the function we want the Neural Network to approximate is the Black-Scholes delta $\Phi(d_1)$, since under a the Black-Scholes simulation, if you could rebalance continuously, the Black-Scholes delta hedging P&L would be zero (perfect hedge).

##### *Advantages*

- Universal approximation theorem of a Neural Network (converges to good answer if set up properly)
- Easily adjustable for transaction costs, market impacts, trading constraints, etc. (already done, lmk if you want to see)
- Can add inputs (option surfaces) to the network for multi-instruments hedging
- Can be adjusted from a hedging prospective to a optimal execution strategy (rebalancing rule in a covered call startegy)

##### *Disadvantages*

- Requires lot of training data.
- Black box: Sometime, the learning agent may 'sacrifice' some paths to optimizes other. 
- Black box: The hedging agent may speculate, or take high risk early on.

___
### Option Surface Modeling

Hans Buehler, Blanka Horvath, Anastais Kratsios, Yannick Limmer, and Raeid Saqur. Sanos - smooth arbitrage-free non-parametric option surfaces. https://arxiv.org/abs/2601.11209, February 2026.


*SANOS (Smooth strictly Arbitrage-free Non-parametric Option Surfaces) represents call prices as a convex combination of Black-Scholes call kernels anchored at a grid of strikes and variances (observed and extended). The weights, a martingale density, are fit via linear/convex programming to match observed market bid/ask prices; because the combination is convex by construction, the resulting surface is automatically smooth and strictly arbitrage-free, and extends naturally to strikes and expiries beyond those directly quoted.*


___
### Option Surface Simulation

Hans Buehler, Blanka Horvath, Anastais Kratsios. Dysanos - Generative Dynamic Smooth Arbitrage-free Non-parametric Option Surfacess. https://arxiv.org/abs/2608.12587, August 2026.

##### *Surface re-modeling*

The above option surface modeling SANOS can be viewed as the mapping
$$
(q, W; \hat{K}, \hat{T}, \mu) \xrightarrow{SANOS} C_{\tiny SANOS}
$$
where $q, W$ are optimized variables, $\hat{K}, \hat{T}, \mu$ are deterministic (surface grid and smoothing parameter). $C_{\tiny SANOS}$ is a complete option surface

Since $q$ is subject to linear constraints, it is hard to use ML on it. The idea of ML-SANOS is to create the mapping
$$
\small 
(x) \xrightarrow{\tiny ML-SANOS} 
(\Sigma, W) \xrightarrow{\tiny DLV} (q) \xrightarrow{\tiny SANOS} C_{\tiny SANOS}
$$
where $x$ and the decoder function ML-SANOS are trained from real market option surfaces. The objective function are the sum of losses from SANOS option surface against bid-ask and mid prices.

##### *Intermediary Results: Option Surface Time Series*


In this new parametrization, $x$ lives in the unrestricted space and is therefore well suited for ML/AI based learning. Note that $(x)$ is already a valid time series which can be decoded to obtain a valid option surface at every time steps.

For efficient time series analysis, reduce the dimension of $(x) \in \mathbb{R}^{\tiny FullDim}$ to $h \in \mathbb{R}^{\tiny LowDim}$

$$
(h)^{\tiny LowDim} \xrightarrow{\tiny NQ} (x)^{\tiny FullDim} 
$$

After training, we are left with 2 things:
- A time serie {${h_t}$}$_{t=1}^{N_{\tiny training}}$, $h_t \in \mathbb{R}^{20}$ for each training day $t$
- A single shared decoder $\theta$ from $NQ_{\theta}$ (that leads to a valid SANOS option surface), valid across all days

Note that in practice, we train the whole thing at once ($x$ and $h$ are train together)

##### *Surface Simulation*

Assume we are given historic samples of ML-SANOS surface states $\tilde{h}_t = (\tilde{h}^1_t, \ldots, \tilde{h}^{n_h}_t) \in \mathbb{R}^{n_h}$ for each historic date $t \in {t_1, \ldots, t_{n_t}}$. We also observe log-spot $\tilde{s}_t := \log S_t$. We use the tilde to distinguish real observed data from simulated data.

<br>

Model a time serie $h$ as followed (baseline model, potential for further research):

*Continuous-time PCA-AR(1)*
$$
dh_t = \kappa (m - h_t) \ dt + \Sigma_h \ dW_t^h
$$
for $m \in \mathbb{R}^{n_h}$, $\kappa \in \mathbb{R}^{n_h, n_h}$, $\Sigma_h \in \mathbb{R}^{n_h,n_\alpha}$ and a Brownian motion $W_t^h \in \mathbb{R}^{n_\alpha}$. $n_h$ is the dimention of thre vector $h_t$, $n_{\alpha}$ is the number of PCA factors (or no PCA and analyse full dimension of $h$).

<br>

Now model the log-spot price time serie $s_t$

*Log-spot* (fitted afterward)
$$
ds_t = \mu dt + \beta^{\prime} dW_t^h + \varsigma dW_t^s
$$
where $\mu \in \mathbb{R}$ is an intercept, $\beta \in \mathbb{R}^{n_h}$, $\varsigma \in \mathbb{R}$ and $dW_t^s \in \mathbb{R}$ is an independant Brownian motion. This preserves the autonomous surface dynamics and captures contemporaneous spot–surface dependence through $\beta$.

*Simulation*

1. Draw randomly $h_0$ from ${\tilde h}_{t\in{1,\dots,n_t}}$ or the invariant distribution.

2. Generate the surface path using the Euler discretization
    $$
    h_{t+dt} = h_t + \kappa(m - h_t)\Delta_t + \Sigma_h Z_t\sqrt{\Delta_t}
    $$
    where $\Delta_t$ is the normalized time increment. The noise term is defined as mixture
    $$
    Z_t = \sqrt{1-b^2} \ \tilde\alpha_{I_t(\omega)} + b \ Y_t,
    $$
    between a resampled standardized historic PCA score and an independent standard normal $Y_t \sim \mathcal{N}(0, I_{n_\alpha})$ for bandwidth $b\in[0,1]$. Here $I_t(\omega)$ is sampled from the indices of the historic transitions. Hence $b=0$ gives empirical innovation resampling, while $b=1$ gives a Gaussian innovation with the same fitted covariance.

3. Conditional on the complete surface path, generate log-spot using 
    $$
    ds_t = \mu dt + \beta^{\prime} dW_t^h + \varsigma dW_t^s
    $$

*Results*

- First, we have the historical time serie {$\tilde{h}_t$}, which can be decoded to complete valid option surfaces at each $t$.

- Second, from this time serie, we can simulate $h_t$ (which can be decoded to option surface). Since the option surfaces are normalized (and more), we can simulate spot prices based on the simulated option surfaces, through a defined (or fitted) dependance.

___
### **Research Proposal**
___

The research proposal is as follow:
- Obtain the historical option surfaces and fit the time series {$\tilde{h}_t$}, which can be directly decoded to the full option surface.
- Analyse this resulting time series (PCA, AR(1), GARCH, Generative AI models, etc.) and simulate jointly option surfaces $h_t$ and spot prices $s_t$.
- Use the deep hedging agent to learn optimal rebalancing (timing, size, moneyness) and manage exposure (delta, gamma, theta), using the simulated option surfaces and spot has inputs. Note that from this joint simulation, the agent can use the option surfaces as an input or for trading other options.

Main challenges:
- Use deep hedging with multi-assets and multi-inputs
- De-americanized option: the above theory is about European option. Trading american option requires an adjustment (called de-americanisation). Since the investment strategy is not naked (mainly covered), the risk from European to American option not as important.
- Efficient time serie analysis (classic time series, generative AI time series)